# Model recall testing: workflow for WildObs Image Management Platform

## Description
- This script can be used for benchmarking an ai species recognition model with a local dataset to get an independent assessment of Recall for a given location
- The purpose is to inform an appropriate validation workflow for species relevant to the monitoring

## Setup instructions:
1) Prepare the testing dataset. Organise camera trap images into folders by species. Suggest using same number of images of each species e.g. 1000.
2) Establish a new project in the WildObs WIMP for benchmarking purposes. Name the project based on the model that will be tested e.g. "Model benchmark testing: WildObs National". You will need to create a separate project for each model.
3) Define tags in the project based on the scientific names of the species you are testing. These need to match with the names used in the WIMP.
4) Configure the project to use the model you want to test
5) Add a deployment to the project for each species and upload the relevant images into each deployment. Use the tags created earlier to assign to the deployment so you know which species it is supposed to be.
6) Run the images through the AI species recognition model
7) Export the project data in Camtrap DP format
8) Extract the data export to a folder
9) Use the folder path as input to this script

In [102]:
import pandas as pd
import os
import re
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


# -------- USER INPUT --------

data_source_location = "Bon Bon Reserve, SA"
target_species = "Feral cat (Felis catus), Red Fox (Vulpes vulpes), European Rabbit (Oryctolagus cuniculus)"
data_collation_process = """
Collated 1000 images of each target species.
Images were manually validated as having at least one detection of the target species and no other species present in the image.
"""

output_report_folder = r"C:\Users\colin.broughton\Downloads\Model_Benchmarking_Report_Exports"

#model_name = "WildObs National"
#camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-wildobs-national-20260318222241"

model_name = 'AWC135'
camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-awc-135-20260318222419"

#model_name = 'SpeciesNet V4'
#camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-speciesnet-v4-20260318221911"

results_interpretation_notes = """
<h4>Intended purpose and use of this report</h4>
<ul>
    <li>Inform the potential suitability of a model for a given location and set of species relevant to the monitoring program.</>
    <li>Inform the design of a validation workflow (manual human review) to ensure a sufficient level of accuracy can be reached to meet the goals of the monitoring program.</li>
    <li>Inform potential gaps in model performance and priorities for provision of additional training images to improve performance.</li>
</ul>

<h4>What is Recall?</h4>
<ul>
    <li>Recall (also called sensitivity or true positive rate) measures how well a model identifies all actual positive cases.</li>
    <li>In simple terms: Of all the true positives, how many did the model correctly find?</li>
    <li>Example: If there are 100 images of Feral Cat within a dataset, and the model correctly detects 90 of them, recall = 0.9 (90%).</li>
    <li>Quantifying recall helps us to understand how many true detections of a given species might be missed by the model.</li>
</ul>
<h4>What is a Confusion Matrix?</h4>
<ul>
    <li>A confusion matrix summarises the model’s performance by comparing actual vs predicted labels.</li>
    <li>In simple terms the confusion matrix shows what the model got right and wrong, broken down by type of error.</li>
    <li>Inspecting the confusion matrix helps us to understand the direction of error in cases where an image was incorrectly classified.</li>
    <li>Be aware that not all models define or name species the same way. In some cases a model may also generalise a species detections to a broader group e.g. Genus or Family. If recall is returning a very low number for a given species and model, inspection of confusion matrix may help to reveal inconsistencies in naming or categorisation.</li>
</ul>
"""


# ----------------------------


# -----------------------------
# Load required tables
# -----------------------------
observations = pd.read_csv(os.path.join(camtrap_folder, "observations.csv"))
media = pd.read_csv(os.path.join(camtrap_folder, "media.csv"))
deployments = pd.read_csv(os.path.join(camtrap_folder, "deployments.csv"))


# -----------------------------
# Keep only required columns
# -----------------------------
observations = observations[["eventID", "deploymentID", "scientificName"]]

media = media[["mediaID", "mediaComments"]]

deployments = deployments[["deploymentID", "deploymentTags"]]


# -----------------------------
# Extract sequenceID from mediaComments
# -----------------------------
def extract_sequence(comment):

    if pd.isna(comment):
        return None

    match = re.search(r"sequenceID:([^\s]+)", str(comment))

    if match:
        return match.group(1)

    return None


media["sequenceID"] = media["mediaComments"].apply(extract_sequence)


# -----------------------------
# Join media → observations
# -----------------------------
merged = pd.merge(
    media,
    observations,
    left_on="sequenceID",
    right_on="eventID",
    how="left"
)


# -----------------------------
# Join observations → deployments
# -----------------------------
merged = pd.merge(
    merged,
    deployments,
    on="deploymentID",
    how="left"
)


# -----------------------------
# Normalize values (clean only, no mapping)
# -----------------------------
merged["true_species"] = merged["deploymentTags"].astype(str).str.strip().str.lower()
merged["pred_species"] = merged["scientificName"].astype(str).str.strip().str.lower()


# -----------------------------
# Remove rows with missing truth
# -----------------------------
merged = merged[merged["true_species"].notna()]


# -----------------------------
# Recall calculation (generic)
# -----------------------------
results = []

species_list = sorted(merged["true_species"].dropna().unique())

for species in species_list:

    subset = merged[merged["true_species"] == species]

    tp = (subset["pred_species"] == species).sum()
    fn = (subset["pred_species"] != species).sum()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    results.append({
        "species": species,
        "detections": len(subset),
        "true_positives": tp,
        "false_negatives": fn,
        "recall": round(recall, 4)
    })


results_df = pd.DataFrame(results)


In [103]:
# Output the results (view within notebook)

print("\n")
print(f"Model recall: {model_name}")

display(results_df)

print("Detection Counts")
display(merged["true_species"].value_counts().rename_axis("species").reset_index(name="count"))

print("Predicted Species Counts")
display(merged["pred_species"].value_counts().rename_axis("species").reset_index(name="count"))

conf_matrix = pd.crosstab(
    merged["true_species"],
    merged["pred_species"]
)

print("Confusion Matrix")
display(conf_matrix)



Model recall: AWC135


,species,detections,true_positives,false_negatives,recall
0,felis catus,1009,929,80,0.9207
1,oryctolagus cuniculus,1011,0,1011,0.0000
2,vulpes vulpes,1007,889,118,0.8828


Detection Counts


,species,count
0,oryctolagus cuniculus,1011
1,felis catus,1009
2,vulpes vulpes,1007


Predicted Species Counts


,species,count
0,lagomorpha,957
1,felis catus,945
2,vulpes vulpes,896
3,nan,84
4,bettongia lesueur,50
5,canis familiaris,18
6,macrotis lagotis,10
7,capra hircus,8
8,wallabia bicolor,7
9,lasiorhinus latifrons,6


Confusion Matrix


pred_species,anas strepera,antechinus flavipes,aves,bettongia lesueur,bos taurus,canis familiaris,capra hircus,cercartetus nanus,climacteris rufus,crocodylus johnstoni,felis catus,homo sapiens,isoodon obesulus,lagomorpha,lagostrophus fasciatus,lasiorhinus latifrons,macropus dorsalis,macrotis lagotis,myrmecobius fasciatus,nan,notamacropus dorsalis,ocyphaps lophotes,onychogalea fraenata,oreoica gutturalis,petrogale brachyotis,petrogale lateralis,phascolarctos cinereus,scincidae,sus scrofa,tachyglossus aculeatus,trichosurus caninus,vulpes vulpes,wallabia bicolor
true_species,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
felis catus,0,0,0,18,1,3,3,0,1,1,929,0,0,2,0,3,0,0,1,28,0,1,2,1,1,1,0,0,1,2,3,6,1
oryctolagus cuniculus,0,0,4,2,0,0,0,1,0,0,7,0,3,954,1,0,1,10,0,21,1,0,0,0,0,0,0,0,0,0,1,1,4
vulpes vulpes,1,1,1,30,2,15,5,0,0,0,9,1,0,1,0,3,2,0,0,35,1,0,3,1,0,0,1,1,0,1,2,889,2


In [104]:
# export report to html file

html_file = output_report_folder + "/" + f"Model_Benchmark_Test_Report_{model_name}.html"

style = """
<style>
body { font-family: Arial; margin: 40px; }
h1 { color: #2c3e50; }
h2 { color: #34495e; margin-top: 30px; }
table { border-collapse: collapse; width: 80%; }
th, td { border: 1px solid #ccc; padding: 8px; text-align: center; }
th { background-color: #f2f2f2; }
</style>
"""

with open(html_file, "w") as f:

    f.write(style)
    f.write(f"<h1>Model Benchmark Test Report: {model_name}</h1>")
    
    f.write(f"<h2>Test Details</h2>")
    f.write(f"<h3>Computer Vision (CV) model tested</h3>")
    f.write(f"<p>{model_name}</p>")
    f.write(f"<h3>Source location of test images</h3>")
    f.write(f"<p>{data_source_location}</p>")
    f.write(f"<h3>Target species tested</h3>")
    f.write(f"<p>{target_species}</p>")
    f.write(f"<h3>Data collation process used</h3>")
    f.write(f"<p>{data_collation_process}</p>")
    f.write(f"<h3>Interpretation of results</h3>")
    f.write(f"<p>{results_interpretation_notes}</p>")
    
    f.write("<h2>Recall Results</h2>")
    f.write(results_df.to_html(index=False))

    f.write("<h2>Confusion Matrix</h2>")
    f.write(pd.crosstab(
        merged["true_species"],
        merged["pred_species"]
    ).to_html())

print(f"Report saved to: {html_file}")

Report saved to: C:\Users\colin.broughton\Downloads\Model_Benchmarking_Report_Exports/Model_Benchmark_Test_Report_AWC135.html
